<a href="https://github.com/N3iKos/SMFactory">
  <img alt="GitHub repo" src="https://img.shields.io/badge/GitHub-6e5494?style=for-the-badge&logo=github&logoColor=white"/>
</a><br>

---
> **SMFactory — Google Colab Notebook** · *Speed is a feature. Efficiency is the standard.*
>
> Run cells **top-to-bottom** on first use. On subsequent sessions, skip directly to **🚀 Launch**.
>
> 🔑 Civitai API Key → [civitai.com/user/account](https://civitai.com/user/account)<br>
> 🤗 Hugging Face Token → [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)

In [ ]:
# @title 🖥️ **WebUI Installer** {"display-mode":"form"}
# @markdown ### Step 1 — Pick your WebUI
Webui = 'A1111' # @param ["A1111", "Forge", "ReForge", "ReForge-old", "Forge-Classic", "Forge-Neo", "ComfyUI", "SwarmUI"]
# @markdown ---
# @markdown ### Step 2 — API Keys *(credentials are runtime-only, never stored)*
# @markdown > 🔑 Get Civitai key → https://civitai.com/user/account
Civitai_Key = '' # @param {type:"string", placeholder:"Your Civitai API Key (required)"}
# @markdown > 🤗 Get HF token → https://huggingface.co/settings/tokens
HF_Read_Token = '' # @param {type:"string", placeholder:"Your Huggingface READ Token (optional)"}
# @markdown ---
# @markdown ### Step 3 — Google Drive *(optional — for persistent model storage)*
# @markdown > If **Yes**, models are symlinked to `MyDrive/Segsmaker/` and survive session resets.
Mount_GDrive = 'No' # @param ["Yes", "No"]

import subprocess, sys
from pathlib import Path

if Mount_GDrive == 'Yes':
    from google.colab import drive
    drive.mount('/content/drive')

# Download setup.py from SMFactory and run installer
_setup_py = '/content/setup.py'
_url = 'https://github.com/N3iKos/SMFactory/raw/main/script/KC/setup.py'
_r = subprocess.run(['curl', '-fsSLo', _setup_py, _url], capture_output=True, text=True)
if _r.returncode != 0:
    print(f'❌ Setup download failed:\n{_r.stderr}')
    sys.exit(1)

get_ipython().run_line_magic('run', f'{_setup_py} --webui="{Webui}" --civitai_key="{Civitai_Key}" --hf_read_token="{HF_Read_Token}"')

# Google Drive symlinks — only if mounted
if Mount_GDrive == 'Yes':
    _drive = Path('/content/drive/MyDrive/Segsmaker')
    for _name, _path in {
        'checkpoint': CKPT,
        'lora':       LORA,
        'vae':        VAE,
        'embeddings': Embeddings
    }.items():
        _folder = _drive / _name
        _folder.mkdir(parents=True, exist_ok=True)
        _sym = _path / f'drive-{_name}'
        if not _sym.exists():
            _sym.symlink_to(_folder, target_is_directory=True)

    import os
    os.system(f'rm -rf {WebUI_Output}')
    _output = _drive / {'ComfyUI': 'comfyui-output', 'SwarmUI': 'swarmui-output'}.get(Webui, 'output')
    _output.mkdir(parents=True, exist_ok=True)
    WebUI_Output.symlink_to(_output, target_is_directory=True)

    if Webui not in {'ComfyUI', 'SwarmUI'}:
        _wc = WebUI / 'cache'
        os.system(f'rm -rf {_wc}')
        _cache = _drive / 'cache'
        _cache.mkdir(parents=True, exist_ok=True)
        _wc.symlink_to(_cache, target_is_directory=True)

In [ ]:
# @title 📥 **Model Downloader** — 5 Checkpoint · 5 LoRA · 1 VAE {"display-mode":"form"}
# @markdown ---
# @markdown ### 🗃️ Checkpoints
# @markdown > Supported sources: `civitai.com`, `huggingface.co`, direct links, Google Drive.
# @markdown > Example: `https://civitai.com/models/133005` or `https://huggingface.co/pantat88/back_up/resolve/main/bigblu25dmix25DStyle_v10.safetensors`
Checkpoint_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Checkpoint_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎨 LoRA
# @markdown > Example: `https://civitai.com/models/122359` ← Detail Tweaker XL
Lora_1 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_2 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_3 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_4 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
Lora_5 = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### 🎛️ VAE
# @markdown > Example: `https://huggingface.co/stabilityai/sd-vae-ft-mse-original/resolve/main/vae-ft-mse-840000-ema-pruned.safetensors`
VAE_URL = '' # @param {type:"string", placeholder:"URL or leave empty to skip"}
# @markdown ---
# @markdown ### ⚡ Download Mode
# @markdown > **Parallel** = download all files at the same time (recommended).
# @markdown > **Max Workers**: 2 = safe for free Colab · 3–4 = faster on Colab Pro.
Parallel_Download = True  # @param {type:"boolean"}
Max_Workers       = 3     # @param {type:"slider", min:1, max:6, step:1}

from nenen88 import parallel_batch_download

_queue = []
for _url in [Checkpoint_1, Checkpoint_2, Checkpoint_3, Checkpoint_4, Checkpoint_5]:
    if _url.strip(): _queue.append((_url.strip(), str(CKPT), None))
for _url in [Lora_1, Lora_2, Lora_3, Lora_4, Lora_5]:
    if _url.strip(): _queue.append((_url.strip(), str(LORA), None))
if VAE_URL.strip(): _queue.append((VAE_URL.strip(), str(VAE), None))

if not _queue:
    print('  No URLs provided — skipping.')
elif Parallel_Download:
    parallel_batch_download(_queue, max_workers=Max_Workers)
else:
    for _url, _dest, _fn in _queue:
        %cd -q $_dest
        %download $_url

In [ ]:
# @title 🛠️ **Extra Assets** — 5 Extensions · 3 Embeddings · 3 Upscalers {"display-mode":"form"}
# @markdown ---
# @markdown ### 🔌 Extensions / Custom Nodes *(git clone URL)*
# @markdown > Example: `https://github.com/Mikubill/sd-webui-controlnet`
Extension_1 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_2 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_3 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_4 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
Extension_5 = '' # @param {type:"string", placeholder:"git clone URL or leave empty"}
# @markdown ---
# @markdown ### 🖼️ Embeddings
# @markdown > Example: `https://huggingface.co/datasets/Nerfgun3/bad_prompt/resolve/main/bad_prompt_version2.pt`
Embedding_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Embedding_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}
# @markdown ---
# @markdown ### 🔬 Upscalers
# @markdown > Example: `https://huggingface.co/ai-forever/Real-ESRGAN/resolve/main/RealESRGAN_x4.pth`
Upscaler_1 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_2 = '' # @param {type:"string", placeholder:"URL or leave empty"}
Upscaler_3 = '' # @param {type:"string", placeholder:"URL or leave empty"}

import os, tempfile
from nenen88 import parallel_batch_download, parallel_clone

# Extensions: parallel shallow clone
_ext_urls = [u.strip() for u in [Extension_1, Extension_2, Extension_3, Extension_4, Extension_5] if u.strip()]
if _ext_urls:
    print('\n⚡ Cloning extensions / custom nodes in parallel...')
    parallel_clone(_ext_urls, str(Extensions), max_workers=5)

# Embeddings + Upscalers: parallel batch download
_asset_queue = []
for _url in [Embedding_1, Embedding_2, Embedding_3]:
    if _url.strip(): _asset_queue.append((_url.strip(), str(Embeddings), None))
for _url in [Upscaler_1, Upscaler_2, Upscaler_3]:
    if _url.strip(): _asset_queue.append((_url.strip(), str(Upscalers), None))
if _asset_queue:
    print('\n📦 Downloading extra assets...')
    parallel_batch_download(_asset_queue, max_workers=3)
elif not _ext_urls:
    print('  No extra assets provided — skipping.')

In [ ]:
# @title ⚡ **FLUX Model Downloader** {"display-mode":"form"}
# @markdown ### Select FLUX Variant
# @markdown > Works with **Forge**, **ComfyUI**, **SwarmUI**. Leave **None** if not needed.
FLUX_Variant = 'None' # @param ["None", "FLUX.1-schnell (Fast, 4-step)", "FLUX.1-dev (Quality, 20-step)"]
# @markdown > - **Schnell** = fastest, permissive Apache 2 license
# @markdown > - **Dev** = best quality, non-commercial license
# @markdown ---
# @markdown ### Component URLs *(FP8 quantized — optimized for Colab VRAM)*
# @markdown > You may replace these with higher-precision versions if you have enough VRAM.
FLUX_Unet   = 'https://huggingface.co/Kijai/flux-fp8/resolve/main/flux1-schnell-fp8.safetensors' # @param {type:"string"}
FLUX_Clip_L = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/clip_l.safetensors' # @param {type:"string"}
FLUX_T5XXL  = 'https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors' # @param {type:"string"}
FLUX_VAE    = 'https://huggingface.co/black-forest-labs/FLUX.1-schnell/resolve/main/ae.safetensors' # @param {type:"string"}
# @markdown ---
# @markdown ### Custom FLUX Checkpoint *(optional — overrides Variant selection)*
# @markdown > Use if you have a specific merged/fine-tuned FLUX .safetensors to download instead.
FLUX_Custom_Checkpoint = '' # @param {type:"string", placeholder:"URL to a .safetensors FLUX checkpoint, or leave empty"}
# @markdown > ⚠️ FLUX is 10–25 GB total. Ensure sufficient Colab disk space before proceeding.

from nenen88 import parallel_batch_download

if FLUX_Custom_Checkpoint.strip():
    print('\n⚡ Downloading custom FLUX checkpoint...')
    parallel_batch_download([(FLUX_Custom_Checkpoint.strip(), str(CKPT), None)], max_workers=1)
elif FLUX_Variant != 'None':
    # Auto-swap schnell → dev unet if dev variant selected
    _unet = FLUX_Unet
    if 'dev' in FLUX_Variant.lower() and 'schnell' in FLUX_Unet:
        _unet = FLUX_Unet.replace('schnell-fp8', 'dev-fp8').replace('flux1-schnell', 'flux1-dev')
    _vae_fn = 'flux_ae.safetensors'
    _flux_queue = [(u, d, f) for u, d, f in [
        (_unet,       str(UNET), None),
        (FLUX_Clip_L, str(CLIP), None),
        (FLUX_T5XXL,  str(CLIP), None),
        (FLUX_VAE,    str(VAE),  _vae_fn),
    ] if u.strip()]
    print(f'\n⚡ Downloading {FLUX_Variant} ({len(_flux_queue)} files in parallel)...')
    parallel_batch_download(_flux_queue, max_workers=2)
    print('\n✅ FLUX ready. Enable FLUX support in your WebUI settings.')
else:
    print('  FLUX_Variant is "None" and no custom checkpoint provided — skipping.')

In [ ]:
# @title 🎛️ **ControlNet Widget**
%run $Controlnet_Widget

In [ ]:
# @title 🚀 **Launch WebUI** {"display-mode":"form"}
# @markdown ### Launch Arguments
# @markdown > Recommended args per WebUI:
# @markdown > - **A1111** → `--xformers`
# @markdown > - **Forge** → `--disable-xformers --opt-sdp-attention --cuda-stream`
# @markdown > - **ReForge** → `--xformers --cuda-stream`
# @markdown > - **Forge-Classic** → `--xformers --cuda-stream --persistent-patches`
# @markdown > - **Forge-Neo** → `--xformers --cuda-malloc --cuda-stream`
# @markdown > - **ComfyUI** → `--dont-print-server --use-pytorch-cross-attention`
# @markdown > - **SwarmUI** → `--launch_mode none`
# @markdown ---
# @markdown > Add `--N=your_ngrok_token` to use NGROK tunnel.<br>
# @markdown > Add `--Z=your_zrok_token` to use ZROK tunnel.
Extra_Args          = '--xformers' # @param {type:"string", placeholder:"e.g. --xformers --medvram"}
Skip_Widget         = False        # @param {type:"boolean"}
# @markdown > Check to skip the launcher widget and go straight to the WebUI.
Skip_ComfyUI_Check  = False        # @param {type:"boolean"}
# @markdown > Check to skip ComfyUI node dependency validation (faster cold start).

_args = Extra_Args.strip()
if Skip_Widget:        _args += ' --skip-widget'
if Skip_ComfyUI_Check: _args += ' --skip-comfyui-check'

%cd -q $WebUI
get_ipython().run_line_magic('run', f'segsmaker.py {_args}')